# Hypotheses recorded before analysis

Recorded on October 13, 2026, before calculating any hypothesis results.

Use the frozen gender-excluded model and its saved blackbox predictions only (61,002 applicants). No retraining, threshold selection, test evaluation or eyeball case inspection. TARGET=1 means recorded repayment difficulty, not necessarily permanent default. The operating cutoff stays 0.121.

H1: EXT_SOURCE_2 < 0.11 has positive mean calibration residual, mean(predicted probability minus TARGET). Primary test uses every blackbox applicant in that group, not only false positives. Report count, observed event rate, mean probability, residual and approximate 95% interval. Predeclare 2 percentage points as an illustrative practical overprediction margin, not a legal or universal standard. Report other EXT_SOURCE_2 bands [0.11,0.25), [0.25,0.5), [0.5,1] and missing separately.

H2: AMT_CREDIT / AMT_INCOME_TOTAL > 4 is associated with more false positives among actual TARGET=0 applicants, even at roughly comparable external scores. Exclude nonpositive/missing income and missing credit from this comparison and report exclusions. Compare >4 versus <=4 at the frozen cutoff. Primary adjustment: simultaneous fixed 0.20-wide bands for EXT_SOURCE_1, EXT_SOURCE_2 and EXT_SOURCE_3, each with a separate missing category. Retain strata with at least 20 actual negatives in each loan-ratio group. Standardize both groups' FPR to the same pooled-negative stratum weights. Report coverage, risk difference and an approximate independent-binomial 95% interval conditional on the retained strata. Sensitivity check: 0.10-wide bands, with the same minimum counts. Report all-applicant event prevalence and calibration by ratio group as context. This is observational adjustment for three scores, not proof the loan ratio causes errors; residual confounding and limited overlap remain.

H3: Under-30 applicants have overpredicted repayment difficulty, beyond their higher observed event rate. Report predicted versus observed rates, mean residual and approximate 95% interval for Under 30, 30–<45, 45–<60 and 60+. Primary contrast: mean residual Under 30 minus mean residual age 30+. Positive overprediction in the youngest group plus a positive contrast supports an age-specific calibration concern. A high FPR alone does not establish miscalibration. Also report within-age reliability tables at fixed prediction bins [0,.05,.10,.15,.20,.30,.50,1], reporting small bins explicitly. Aggregate calibration can conceal within-bin miscalibration and neither proves the cause of FPR disparities nor excludes other causes.

Uncertainty: residual intervals use sample standard errors of p-y with normal approximation; H2 uses independent-binomial stratum variances and fixed pooled weights. All intervals are descriptive and assume independent applicants, conditioning on the fitted model. No confirmatory p-values or multiple-testing claims. Blackbox data already selected checkpoint and cutoff, hypotheses originate in prior eyeball review, and repeated dev use limits confirmatory interpretation. No causal claim or automatic model change follows any result.


In [1]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
from IPython.display import display

ROOT=Path.cwd()
OUT=ROOT/'hypothesis_analysis'/'blackbox_no_gender'
protocol_metadata=json.loads((OUT/'protocol_metadata.json').read_text())
MODEL=ROOT/protocol_metadata['model_run']
info=json.loads((MODEL/'model_info.json').read_text())
assert info['excluded_predictors']==['CODE_GENDER'] and info['threshold']==.121
pred=pd.read_json(MODEL/'validation_predictions.json')
eye_ids=set(json.loads((MODEL/'dev_split_manifest.json').read_text())['eyeball_ids'])
assert len(pred)==61002 and pred.SK_ID_CURR.is_unique
assert set(pred.SK_ID_CURR).isdisjoint(eye_ids)
columns=['SK_ID_CURR','TARGET','EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3','AMT_CREDIT','AMT_INCOME_TOTAL','DAYS_BIRTH']
raw=pd.read_csv(ROOT/'application_train.csv',usecols=columns)
d=pred.merge(raw,on='SK_ID_CURR',validate='one_to_one')
assert d.actual_TARGET.eq(d.TARGET).all() and d.predicted_probability.between(0,1).all()
d['residual']=d.predicted_probability-d.TARGET
d['flagged']=d.predicted_probability.ge(info['threshold'])
d['age']=-d.DAYS_BIRTH/365.25
z=1.959963984540054

def calibration(g):
    n=len(g);r=g.residual
    se=r.std(ddof=1)/np.sqrt(n) if n>1 else np.nan
    neg=g.TARGET.eq(0);pos=g.TARGET.eq(1)
    return dict(n=n,events=int(pos.sum()),mean_prediction=g.predicted_probability.mean(),
        observed_rate=g.TARGET.mean(),calibration_gap=r.mean(),gap_lower=r.mean()-z*se,gap_upper=r.mean()+z*se,
        false_positive_rate=g.loc[neg,'flagged'].mean(),recall=g.loc[pos,'flagged'].mean(),
        negatives=int(neg.sum()),false_positives=int(g.loc[neg,'flagged'].sum()),small_group=n<100)

def save_table(name,rows):
    frame=pd.DataFrame(rows)
    frame.to_json(OUT/(name+'.json'),orient='records',indent=2)
    print('\n'+name)
    display(frame.round(5))
    return frame

h1=save_table('H1_external_score_calibration',[
    dict(group=name,**calibration(d[mask])) for name,mask in [
        ('EXT2 < 0.11',d.EXT_SOURCE_2.lt(.11)),
        ('0.11 <= EXT2 < 0.25',d.EXT_SOURCE_2.ge(.11)&d.EXT_SOURCE_2.lt(.25)),
        ('0.25 <= EXT2 < 0.50',d.EXT_SOURCE_2.ge(.25)&d.EXT_SOURCE_2.lt(.5)),
        ('EXT2 >= 0.50',d.EXT_SOURCE_2.ge(.5)),
        ('EXT2 missing',d.EXT_SOURCE_2.isna())]])

valid=d.AMT_INCOME_TOTAL.gt(0)&np.isfinite(d.AMT_INCOME_TOTAL)&np.isfinite(d.AMT_CREDIT)
ratio=d.loc[valid].copy()
ratio['credit_income_ratio']=ratio.AMT_CREDIT/ratio.AMT_INCOME_TOTAL
ratio['high_ratio']=ratio.credit_income_ratio.gt(4)
h2_context=save_table('H2_ratio_context',[dict(group=name,**calibration(g)) for name,g in [
    ('Credit/income <= 4',ratio[~ratio.high_ratio]),('Credit/income > 4',ratio[ratio.high_ratio])]])

def adjusted_comparison(width):
    neg=ratio[ratio.TARGET.eq(0)].copy()
    bands=[]
    for c in ['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']:
        col=c+'_band';bands.append(col)
        assert neg[c].dropna().between(0,1).all()
        neg[col]=np.floor(neg[c].clip(upper=1-1e-12)/width).fillna(-1).astype(int)
    grouped=neg.groupby(bands+['high_ratio']).flagged.agg(['size','sum']).unstack('high_ratio')
    # Reindex makes absent ratio groups explicit before restricting to overlap.
    grouped=grouped.reindex(columns=pd.MultiIndex.from_product([['size','sum'],[False,True]]))
    keep=(grouped['size'][False]>=20)&(grouped['size'][True]>=20)
    t=grouped.loc[keep].copy()
    assert len(t)>0,'No common-support strata at this width.'
    n0=t['size'][False];n1=t['size'][True];p0=t['sum'][False]/n0;p1=t['sum'][True]/n1
    w=(n0+n1)/(n0+n1).sum()
    diff=float((w*(p1-p0)).sum())
    se=np.sqrt((w**2*(p0*(1-p0)/n0+p1*(1-p1)/n1)).sum())
    details=t.index.to_frame(index=False)
    for key,val in [('n_low_ratio',n0),('n_high_ratio',n1),('fpr_low_ratio',p0),('fpr_high_ratio',p1),('weight',w)]:
        details[key]=val.to_numpy()
    details.to_json(OUT/f'H2_strata_width_{width:.2f}.json',orient='records',indent=2)
    return dict(band_width=width,strata_retained=len(t),eligible_negatives=len(neg),
        retained_negatives=int((n0+n1).sum()),coverage=float((n0+n1).sum()/len(neg)),
        coverage_low_ratio=float(n0.sum()/(~neg.high_ratio).sum()),coverage_high_ratio=float(n1.sum()/neg.high_ratio.sum()),
        fpr_low_ratio=float((w*p0).sum()),fpr_high_ratio=float((w*p1).sum()),
        adjusted_fpr_difference=diff,difference_lower=diff-z*se,difference_upper=diff+z*se)
h2=save_table('H2_score_adjusted_FPR',[adjusted_comparison(.2),adjusted_comparison(.1)])

d['age_group']=pd.cut(d.age,[0,30,45,60,np.inf],right=False,labels=['Under 30','30 to under 45','45 to under 60','60 and older'])
assert d.age_group.notna().all()
h3=save_table('H3_age_calibration',[dict(group=str(name),**calibration(g)) for name,g in d.groupby('age_group',observed=True)])
young=d[d.age.lt(30)].residual;older=d[d.age.ge(30)].residual
delta=young.mean()-older.mean();se=np.sqrt(young.var(ddof=1)/len(young)+older.var(ddof=1)/len(older))
h3_contrast=dict(young_gap=float(young.mean()),older_gap=float(older.mean()),difference=float(delta),
    lower=float(delta-z*se),upper=float(delta+z*se))
(OUT/'H3_young_older_contrast.json').write_text(json.dumps(h3_contrast,indent=2))
d['probability_bin']=pd.cut(d.predicted_probability,[0,.05,.10,.15,.20,.30,.50,1],include_lowest=True,right=False)
# Include the mathematically possible exact endpoint p=1 in the last bin.
if d.predicted_probability.eq(1).any():d.loc[d.predicted_probability.eq(1),'probability_bin']=d.probability_bin.cat.categories[-1]
reliability=save_table('H3_age_reliability',[
    dict(age_group=str(age),probability_bin=str(b),**calibration(g))
    for (age,b),g in d.groupby(['age_group','probability_bin'],observed=True)])
overall=calibration(d)
(OUT/'overall_calibration.json').write_text(json.dumps(overall,indent=2,default=float))

# Plot existing results; no fitted calibration transformation is applied.
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(7,5),layout='constrained')
for age,g in reliability.groupby('age_group',sort=False):
    g=g[g.n>=100]
    ax.plot(g.mean_prediction,g.observed_rate,'o-',label=age)
ax.plot([0,1],[0,1],'--',color='gray',label='Perfect calibration')
ax.set(xlim=(0,.6),ylim=(0,.6),xlabel='Mean predicted repayment-difficulty probability',ylabel='Observed repayment-difficulty rate',title='Blackbox calibration by age (bins with at least 100 applicants)')
ax.legend();fig.savefig(OUT/'age_calibration.png',dpi=160);display(fig);plt.close(fig)

a=h1.iloc[0];b=h2.iloc[0];s=h2.iloc[1];y=h3.iloc[0]
def interval(lo,hi):return f'[{100*lo:.2f}, {100*hi:.2f}] percentage points'
def evidence(value,lo,hi):
    return 'positive gap, with the descriptive interval above zero' if lo>0 else ('negative gap, with the descriptive interval below zero' if hi<0 else 'interval includes zero')
summary=f'''# Blackbox tests of error-analysis hypotheses

Protocol was recorded before analysis in Protocol.md. Frozen model: {info['source_run']}. Only {len(d):,} blackbox applicants were used. TARGET means repayment difficulty. No model, threshold, or error-review CSV was changed.

## H1: Overprediction for EXT_SOURCE_2 below 0.11

N={int(a.n):,}; observed difficulty {a.observed_rate:.2%}; mean prediction {a.mean_prediction:.2%}. Predicted minus observed: {a.calibration_gap*100:.2f} percentage points, approximate 95% interval {interval(a.gap_lower,a.gap_upper)}. Result: {evidence(a.calibration_gap,a.gap_lower,a.gap_upper)}. The predeclared illustrative practical margin is +2 percentage points; a point estimate below this is not evidence of large overprediction. False positives alone cannot establish miscalibration because most members of even a risky group may have no difficulty.

**H1 is not supported: the observed direction is underprediction, not overprediction.**

## H2: High credit-to-income ratio and false positives at similar external scores

Excluded {int((~valid).sum())} applicants with invalid/missing ratio inputs. Primary score-band width 0.20: standardized FPR is {b.fpr_low_ratio:.2%} for ratio <=4 and {b.fpr_high_ratio:.2%} for ratio >4. Adjusted difference: {b.adjusted_fpr_difference*100:.2f} points, approximate 95% interval {interval(b.difference_lower,b.difference_upper)}. Result: {evidence(b.adjusted_fpr_difference,b.difference_lower,b.difference_upper)}. Included {int(b.retained_negatives):,} of {int(b.eligible_negatives):,} eligible actual negatives ({b.coverage:.1%}) across {int(b.strata_retained)} common-support strata. Retention is {b.coverage_low_ratio:.1%} of low-ratio negatives and {b.coverage_high_ratio:.1%} of high-ratio negatives.

At narrower 0.10 score bands, difference={s.adjusted_fpr_difference*100:.2f} points, interval {interval(s.difference_lower,s.difference_upper)}, with {s.coverage:.1%} negative coverage. These are standardized associations within retained strata, not causal effects. Both groups use the same pooled-stratum weights. Other variables and within-band score differences can confound the association. Missing scores form their own bands; their unobserved values are not held constant.

**H2 has modest descriptive support after score adjustment.** Before adjustment, FPR is {h2_context.iloc[0].false_positive_rate:.2%} for ratio <=4 and {h2_context.iloc[1].false_positive_rate:.2%} for ratio >4: the direction reverses. This is why raw counts and unadjusted rates are insufficient. Narrower bands reduce both the difference and overlap coverage, so the sensitivity change is not attributable solely to finer score control. This does not establish that high loan ratios are erroneous signals or should be removed.

## H3: Younger false alarms and calibration

Under 30: N={int(y.n):,}, observed difficulty {y.observed_rate:.2%}, mean prediction {y.mean_prediction:.2%}. Calibration residual {y.calibration_gap*100:.2f} points, interval {interval(y.gap_lower,y.gap_upper)}. Under-30 minus age-30+ calibration residual is {h3_contrast['difference']*100:.2f} points, interval {interval(h3_contrast['lower'],h3_contrast['upper'])}.

The age table and fixed-bin reliability table separate average calibration from FPR. Aggregate agreement does not guarantee calibration in every probability bin. A high false-positive rate may persist even with calibrated probabilities and different risk distributions. This diagnostic does not decompose the age FPR gap into base rates, calibration, and discrimination, so it cannot prove that only one explains the disparity.

**H3 is not supported as an overprediction explanation for younger false alarms.** All four age groups have negative average calibration residuals. The younger-minus-older interval includes zero. Under-30 bins with at least 100 applicants also all have mean predictions below observed rates; small bins are too uncertain for strong conclusions. This does not establish that base rates alone explain the FPR gap.

## Limits and next decisions

Overall mean prediction {overall['mean_prediction']:.2%}, observed difficulty {overall['observed_rate']:.2%}. Hypotheses were motivated by eyeball errors and evaluated using blackbox aggregates. The blackbox set already selected checkpoint and threshold; intervals condition on the fitted model and do not correct for model selection, repeated dev use, multiple hypotheses or possible applicant dependence. Treat the results as exploratory evidence, not independent confirmation or causal proof. No individual blackbox cases were displayed or added to manual review. No test predictions were loaded.

H1/H3 concern probability calibration, not accuracy or a decision cutoff. H2 concerns false alarms among actual negatives; context tables also show prevalence and calibration among all applicants. Any subsequent model change should be evaluated as a separate planned experiment, not selected to remove these particular errors.
'''
(OUT/'Findings.md').write_text(summary)
(OUT/'provenance.json').write_text(json.dumps(dict(model_run=info['source_run'],n=len(d),threshold=info['threshold'],
    protocol_sha256=hashlib.sha256((OUT/'Protocol.md').read_bytes()).hexdigest(),
    predictions_sha256=hashlib.sha256((MODEL/'validation_predictions.json').read_bytes()).hexdigest(),
    loaded_prediction_file='validation_predictions.json',eyeball_excluded=True,test_predictions_loaded=False),indent=2))
print(summary)



H1_external_score_calibration

H2_ratio_context

H2_score_adjusted_FPR

H3_age_calibration

H3_age_reliability
# Blackbox tests of error-analysis hypotheses

Protocol was recorded before analysis in Protocol.md. Frozen model: application_no_gender_weight_decay_20261013T064129748622Z. Only 61,002 blackbox applicants were used. TARGET means repayment difficulty. No model, threshold, or error-review CSV was changed.

## H1: Overprediction for EXT_SOURCE_2 below 0.11

N=2,537; observed difficulty 23.02%; mean prediction 18.26%. Predicted minus observed: -4.76 percentage points, approximate 95% interval [-6.34, -3.18] percentage points. Result: negative gap, with the descriptive interval below zero. The predeclared illustrative practical margin is +2 percentage points; a point estimate below this is not evidence of large overprediction. False positives alone cannot establish miscalibration because most members of even a risky group may have no difficulty.

**H1 is not supported: the observ

Matplotlib is building the font cache; this may take a moment.


,group,n,events,mean_prediction,observed_rate,calibration_gap,gap_lower,gap_upper,false_positive_rate,recall,negatives,false_positives,small_group
0,EXT2 < 0.11,2537,584,0.18260,0.23019,-0.04759,-0.06343,-0.03176,0.59652,0.81336,1953,1165,False
1,0.11 <= EXT2 < 0.25,4865,762,0.12818,0.15663,-0.02845,-0.03829,-0.01861,0.38387,0.68373,4103,1575,False
2,0.25 <= EXT2 < 0.50,15712,1506,0.08546,0.09585,-0.01039,-0.01487,-0.00590,0.20731,0.48141,14206,2945,False
3,EXT2 >= 0.50,37750,2062,0.04208,0.05462,-0.01254,-0.01478,-0.01030,0.04719,0.19787,35688,1684,False
4,EXT2 missing,138,11,0.04838,0.07971,-0.03133,-0.07654,0.01388,0.03150,0.00000,127,4,False


,group,n,events,mean_prediction,observed_rate,calibration_gap,gap_lower,gap_upper,false_positive_rate,recall,negatives,false_positives,small_group
0,Credit/income <= 4,37586,3089,0.06675,0.08218,-0.01543,-0.01810,-0.01277,0.13621,0.44124,34497,4699,False
1,Credit/income > 4,23416,1836,0.06475,0.07841,-0.01366,-0.01698,-0.01034,0.12391,0.41721,21580,2674,False


,band_width,strata_retained,eligible_negatives,retained_negatives,coverage,coverage_low_ratio,coverage_high_ratio,fpr_low_ratio,fpr_high_ratio,adjusted_fpr_difference,difference_lower,difference_upper
0,0.2,105,56077,54720,0.97580,0.97287,0.98049,0.12027,0.13633,0.01607,0.01158,0.02055
1,0.1,222,56077,45769,0.81618,0.80262,0.83786,0.10042,0.10641,0.00598,0.00159,0.01037


,group,n,events,mean_prediction,observed_rate,calibration_gap,gap_lower,gap_upper,false_positive_rate,recall,negatives,false_positives,small_group
0,Under 30,8900,1007,0.09616,0.11315,-0.01699,-0.02329,-0.01069,0.24984,0.58093,7893,1972,False
1,30 to under 45,24675,2246,0.07184,0.09102,-0.01918,-0.02262,-0.01574,0.15208,0.46527,22429,3411,False
2,45 to under 60,20531,1340,0.05627,0.06527,-0.00900,-0.01228,-0.00571,0.09405,0.33582,19191,1805,False
3,60 and older,6896,332,0.03498,0.04814,-0.01316,-0.01814,-0.00819,0.02818,0.14759,6564,185,False


,age_group,probability_bin,n,events,mean_prediction,observed_rate,calibration_gap,gap_lower,gap_upper,false_positive_rate,recall,negatives,false_positives,small_group
0,Under 30,"[0.0, 0.05)",3070,101,0.02832,0.03290,-0.00458,-0.01088,0.00172,0.00000,0.00000,2969,0,False
1,Under 30,"[0.05, 0.1)",2527,210,0.07271,0.08310,-0.01039,-0.02113,0.00035,0.00000,0.00000,2317,0,False
2,Under 30,"[0.1, 0.15)",1567,243,0.12314,0.15507,-0.03193,-0.04988,-0.01399,0.52039,0.54321,1324,689,False
3,Under 30,"[0.15, 0.2)",809,158,0.17208,0.19530,-0.02322,-0.05054,0.00411,1.00000,1.00000,651,651,False
4,Under 30,"[0.2, 0.3)",665,186,0.23947,0.27970,-0.04023,-0.07425,-0.00620,1.00000,1.00000,479,479,False
5,Under 30,"[0.3, 0.5)",258,107,0.35517,0.41473,-0.05956,-0.11909,-0.00003,1.00000,1.00000,151,151,False
6,Under 30,"[0.5, 1.0)",4,2,0.52167,0.50000,0.02167,-0.53795,0.58128,1.00000,1.00000,2,2,True
7,30 to under 45,"[0.0, 0.05)",12902,461,0.02355,0.03573,-0.01218,-0.01537,-0.00900,0.00000,0.00000,12441,0,False
8,30 to under 45,"[0.05, 0.1)",5957,567,0.07166,0.09518,-0.02352,-0.03096,-0.01609,0.00000,0.00000,5390,0,False
9,30 to under 45,"[0.1, 0.15)",2741,404,0.12261,0.14739,-0.02479,-0.03804,-0.01153,0.49208,0.57178,2337,1150,False


<Figure size 700x500 with 1 Axes>

# Additional hypotheses recorded before their blackbox calculations

October 13, 2026. Existing H1–H3 results have already been seen and are not newly preregistered. This extension concerns apparent categories in the current 95-error CSV, not an exhaustive search over every feature or cutoff. No model or threshold changes.

H4 Older high-score missed difficulty: among actual TARGET=1, compare FNR for age>=45 and EXT_SOURCE_2>=0.50 with all other positives. Secondary contrasts: older versus younger within high EXT2, and high versus lower EXT2 among older positives. Missing scores are excluded from the latter contrast. This distinguishes the combined pattern from a uniquely age-related effect.
H5 Younger low-score false alarms: among actual TARGET=0, compare FPR for age<30 and EXT_SOURCE_3<0.25 with all other negatives. Secondary contrasts: young versus older within low EXT3, and low versus higher EXT3 among younger negatives. Missing scores excluded from latter contrast. This tests the error pattern separately from H3's calibration explanation.
H6 Missing external scores: compare any EXT_SOURCE_1/2/3 missing versus all present using overall error rate, and separately FPR and FNR within actual classes. A raw count of missing-score errors is insufficient. No claim of causality or imputation defect.
H7 Near-threshold errors: compare error rates within absolute score distance <=0.02 of frozen 0.121 cutoff versus farther away. Also report what fraction of all errors lies in that band. A majority explanation requires more than 50% of errors in the band. Association with uncertainty is partly expected by definition and does not imply that changing threshold fixes both kinds of error.
H8 Conflicting external scores: where EXT2 and EXT3 are observed, define mixed as one>=0.50 and the other<0.25. Compare overall errors and class-specific FPR/FNR versus other complete EXT2/EXT3 cases. This derives from individual case notes, not a previously established dominant category. Different underlying risk distributions can explain differences.
H9 Employment placeholder: raw DAYS_EMPLOYED=365243 versus other raw values; compare errors, FPR, FNR. This is a weak hypothesis from case notes. The existing preprocessing already corrects the sentinel and adds an indicator; an association cannot establish an uncorrected dataset defect.

For each independent two-group rate comparison, report counts, rates and difference with a descriptive approximate 95% interval, using Wilson group intervals and the Newcombe combination for their difference. Group denominators must match the metric. Hypotheses are exploratory, multiple and based on prior dev work; intervals do not correct for selection or repeated use. Small groups remain limited. Retain fixed earlier cutoffs, display aggregates only, never inspect individual blackbox cases. Existing eyeball data may be used to document the motivation for H1–H9. No test predictions loaded.


In [2]:
def wilson(k,n):
    if not n:return (np.nan,np.nan)
    r=k/n;den=1+z*z/n;center=(r+z*z/(2*n))/den
    half=z*np.sqrt(r*(1-r)/n+z*z/(4*n*n))/den
    return center-half,center+half

rows=[]
def compare(hypothesis,label,data,eligible,flag,metric):
    base=data.loc[eligible].copy();mask=flag.loc[base.index]
    outcome=(base.flagged.ne(base.TARGET)) if metric=='error_rate' else (base.flagged if metric=='FPR' else ~base.flagged)
    a=outcome[mask];b=outcome[~mask];na=len(a);nb=len(b);ka=int(a.sum());kb=int(b.sum())
    pa=ka/na if na else np.nan;pb=kb/nb if nb else np.nan
    la,ua=wilson(ka,na);lb,ub=wilson(kb,nb)
    delta=pa-pb
    rows.append(dict(hypothesis=hypothesis,comparison=label,metric=metric,n_group=na,errors_group=ka,rate_group=pa,
        n_reference=nb,errors_reference=kb,rate_reference=pb,difference=delta,
        lower=delta-np.sqrt((pa-la)**2+(ub-pb)**2),upper=delta+np.sqrt((ua-pa)**2+(pb-lb)**2)))

def evaluate(frame,population):
    start=len(rows);pos=frame.TARGET.eq(1);neg=frame.TARGET.eq(0);allrows=pd.Series(True,index=frame.index)
    older=frame.age.ge(45);high=frame.EXT_SOURCE_2.ge(.5);young=frame.age.lt(30);low=frame.EXT_SOURCE_3.lt(.25)
    compare('H4','Older high EXT2 versus other positives',frame,pos,older&high,'FNR')
    compare('H4','Older versus younger within high EXT2 positives',frame,pos&high,older,'FNR')
    compare('H4','High versus lower EXT2 within older positives',frame,pos&older&frame.EXT_SOURCE_2.notna(),high,'FNR')
    compare('H5','Younger low EXT3 versus other negatives',frame,neg,young&low,'FPR')
    compare('H5','Younger versus older within low EXT3 negatives',frame,neg&low,young,'FPR')
    compare('H5','Low versus higher EXT3 within young negatives',frame,neg&young&frame.EXT_SOURCE_3.notna(),low,'FPR')
    flags={
        'H6':frame[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']].isna().any(axis=1),
        'H7':(frame.predicted_probability-info['threshold']).abs().le(.02),
        'H8':(frame.EXT_SOURCE_2.ge(.5)&frame.EXT_SOURCE_3.lt(.25))|(frame.EXT_SOURCE_3.ge(.5)&frame.EXT_SOURCE_2.lt(.25)),
        'H9':frame.DAYS_EMPLOYED.eq(365243)}
    names={'H6':'Any missing external score versus complete','H7':'Near cutoff versus farther away','H8':'Mixed EXT2 EXT3 versus other observed pairs','H9':'Employment sentinel versus other values'}
    for h,flag in flags.items():
        elig=frame.EXT_SOURCE_2.notna()&frame.EXT_SOURCE_3.notna() if h=='H8' else allrows
        for metric,mask in [('error_rate',elig),('FPR',elig&neg),('FNR',elig&pos)]:
            compare(h,names[h],frame,mask,flag,metric)
    for r in rows[start:]:r['population']=population

# Join the extra audit attribute without changing membership or predictions.
employment=pd.read_csv(ROOT/'application_train.csv',usecols=['SK_ID_CURR','DAYS_EMPLOYED'])
d=d.merge(employment,on='SK_ID_CURR',validate='one_to_one')
evaluate(d,'blackbox')
review_dir=Path(json.loads((ROOT/'active_error_analysis_outputs.json').read_text())['output_dir'])
ey=pd.read_csv(review_dir/'eyeball_predictions.csv')
assert len(ey)==500 and set(ey.SK_ID_CURR)==eye_ids
ey['age']=-ey.DAYS_BIRTH/365.25;ey['flagged']=ey.predicted_probability.ge(info['threshold'])
ey['residual']=ey.predicted_probability-ey.TARGET
evaluate(ey,'eyeball')
additional=pd.DataFrame(rows)
additional.to_json(OUT/'additional_hypothesis_results.json',orient='records',indent=2)
display(additional[additional.population.eq('blackbox')].round(4))
motivation=[]
for name,flag in [('H1',ey.EXT_SOURCE_2.lt(.11)),('H2',ey.AMT_CREDIT/ey.AMT_INCOME_TOTAL>4),('H3',ey.age.lt(30))]:
    g=ey[flag]
    motivation.append(dict(hypothesis=name,**calibration(g)))
pd.DataFrame(motivation).to_json(OUT/'original_hypothesis_eyeball_context.json',orient='records',indent=2)
shares={}
for name,f in [('eyeball',ey),('blackbox',d)]:
    err=f.flagged.ne(f.TARGET);near=(f.predicted_probability-info['threshold']).abs().le(.02)
    shares[name]=dict(errors=int(err.sum()),near_errors=int((err&near).sum()),share=float((err&near).sum()/err.sum()))
(OUT/'near_threshold_error_shares.json').write_text(json.dumps(shares,indent=2))
assert len(d)==61002 and set(d.SK_ID_CURR).isdisjoint(eye_ids)
assert len(ey)==500 and ey.flagged.ne(ey.TARGET).sum()==95
print('Near-threshold shares:',shares)


Near-threshold shares: {'eyeball': {'errors': 95, 'near_errors': 20, 'share': 0.21052631578947367}, 'blackbox': {'errors': 10169, 'near_errors': 2276, 'share': 0.2238174845117514}}


<ipython-input-2-93ba524d65cf>:47: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ey['age']=-ey.DAYS_BIRTH/365.25;ey['flagged']=ey.predicted_probability.ge(info['threshold'])
<ipython-input-2-93ba524d65cf>:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ey['residual']=ey.predicted_probability-ey.TARGET


,hypothesis,comparison,metric,n_group,errors_group,rate_group,n_reference,errors_reference,rate_reference,difference,lower,upper,population
0,H4,Older high EXT2 versus other positives,FNR,785,697,0.8879,4140,2099,0.5070,0.3809,0.3525,0.4062,blackbox
1,H4,Older versus younger within high EXT2 positives,FNR,785,697,0.8879,1277,957,0.7494,0.1385,0.1053,0.1702,blackbox
2,H4,High versus lower EXT2 within older positives,FNR,785,697,0.8879,884,473,0.5351,0.3528,0.3123,0.3915,blackbox
3,H5,Younger low EXT3 versus other negatives,FPR,1037,613,0.5911,55040,6760,0.1228,0.4683,0.4380,0.4980,blackbox
4,H5,Younger versus older within low EXT3 negatives,FPR,1037,613,0.5911,3736,1818,0.4866,0.1045,0.0703,0.1381,blackbox
5,H5,Low versus higher EXT3 within young negatives,FPR,1037,613,0.5911,4774,799,0.1674,0.4238,0.3917,0.4550,blackbox
6,H6,Any missing external score versus complete,error_rate,39269,6886,0.1754,21733,3283,0.1511,0.0243,0.0182,0.0303,blackbox
7,H6,Any missing external score versus complete,FPR,35963,5065,0.1408,20114,2308,0.1147,0.0261,0.0204,0.0317,blackbox
8,H6,Any missing external score versus complete,FNR,3306,1821,0.5508,1619,975,0.6022,-0.0514,-0.0805,-0.0220,blackbox
9,H7,Near cutoff versus farther away,error_rate,5238,2276,0.4345,55764,7893,0.1415,0.2930,0.2793,0.3067,blackbox
